In [1]:
import numpy as np
import pandas as pd

energy = energy = pd.read_parquet('../Data/cleaned_data/dam.parquet') #change to model produced schedule
prices = energy['lmp'].iloc[:288].to_numpy() #numaric array of prices
n = len(prices) #length of array

In [2]:
dp = np.full((n+1, 2), -np.inf) #table to keep track of each state at thee end of each interval
previous = np.full((n+1, 2), -1, dtype=int) #table of prevouis interval state
dp[0,0] = 0 #starting out with nothing

In [3]:
for i in range(n):
    price = prices[i] #current price we are working with

    #best way to finish interval i with an empty
    hold_empty = dp[i,0] 
    sell = dp[i,1] + price

    if sell > hold_empty:
        dp[i+1, 0] = sell
        previous[i+1, 0] = 1
    else:
        dp[i+1, 0] = hold_empty
        previous[i+1, 0] = 0

    #best way to finish interval i with a full battary
    hold_full = dp[i,1]
    buy = dp[i, 0] - price

    if buy > hold_full:
        dp[i+1, 1] = buy
        previous[i+1, 1] = 0
    else:
        dp[i+1, 1] = hold_full
        previous[i+1, 1] = 1

In [4]:
final_state = int(np.argmax(dp[n]))
best_cash_profit = float(dp[n, final_state])
actions = ['Hold'] * n
battery_levels = [0] * n
state = final_state

for i in range(n, 0, -1):
    prev_state = previous[i, state]
    if prev_state == 0 and state ==1:
        actions[i-1]='Buy'
    elif prev_state == 1 and state == 0:
        actions[i-1] = 'Sell'

    battery_levels[i-1] = state
    state = prev_state

In [5]:
schedule = pd.DataFrame({
    'lmp': prices,
    'action': actions,
    'battery_mwh': battery_levels
})

schedule['cash_flow'] = np.where(
    schedule['action']=='Buy', -schedule['lmp'],
    np.where(schedule['action']=='Sell', schedule['lmp'], 0.0)
)
schedule['cumalative_cash'] = schedule['cash_flow'].cumsum()

print(schedule.head(20))
print('Best cash profit:', best_cash_profit)
print('Schedule cash profit:', schedule['cash_flow'].sum())
print('Buys:', actions.count('Buy'))
print('Sells:', actions.count('Sell'))
print('Ending Battery (MWh):', final_state)

assert np.isclose(schedule['cash_flow'].sum(), best_cash_profit)

      lmp action  battery_mwh  cash_flow  cumalative_cash
0   26.92   Hold            0       0.00             0.00
1   23.70    Buy            1     -23.70           -23.70
2   23.86   Sell            0      23.86             0.16
3   23.62   Hold            0       0.00             0.16
4   23.50   Hold            0       0.00             0.16
5   22.21   Hold            0       0.00             0.16
6   21.88   Hold            0       0.00             0.16
7   21.20    Buy            1     -21.20           -21.04
8   21.56   Hold            1       0.00           -21.04
9   23.57   Hold            1       0.00           -21.04
10  24.22   Sell            0      24.22             3.18
11  24.17    Buy            1     -24.17           -20.99
12  25.69   Sell            0      25.69             4.70
13  24.63   Hold            0       0.00             4.70
14  23.87   Hold            0       0.00             4.70
15  22.44    Buy            1     -22.44           -17.74
16  22.83   Se